# 🎓 Prediksi Kelulusan Siswa - Machine Learning
**Mata Kuliah:** Kecerdasan Buatan  
**Algoritma:** Random Forest Classifier  
**Dataset:** Sintetis (500 data siswa)

## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve
)

print('✅ Semua library berhasil diimport!')

## 2. Buat Dataset

In [ ]:
np.random.seed(42)
n = 500

df = pd.DataFrame({
    'age':        np.random.randint(15, 22, n),
    'Medu':       np.random.randint(0, 5, n),
    'Fedu':       np.random.randint(0, 5, n),
    'traveltime': np.random.randint(1, 5, n),
    'studytime':  np.random.randint(1, 5, n),
    'failures':   np.random.randint(0, 4, n),
    'famrel':     np.random.randint(1, 6, n),
    'freetime':   np.random.randint(1, 6, n),
    'goout':      np.random.randint(1, 6, n),
    'Dalc':       np.random.randint(1, 6, n),
    'Walc':       np.random.randint(1, 6, n),
    'health':     np.random.randint(1, 6, n),
    'absences':   np.random.randint(0, 30, n),
    'sex':        np.random.choice(['M', 'F'], n),
    'address':    np.random.choice(['U', 'R'], n),
    'Pstatus':    np.random.choice(['T', 'A'], n),
    'schoolsup':  np.random.choice(['yes', 'no'], n),
    'famsup':     np.random.choice(['yes', 'no'], n),
    'paid':       np.random.choice(['yes', 'no'], n),
    'internet':   np.random.choice(['yes', 'no'], n),
})

# Label kelulusan berdasarkan logika realistis
df['pass'] = (
    (df['studytime'] >= 3) &
    (df['failures'] == 0) &
    (df['absences'] < 15) &
    (df['Medu'] >= 2)
).astype(int)

# Tambah noise 15%
noise = np.random.random(n) < 0.15
df['pass'] = df['pass'].where(~noise, 1 - df['pass'])

# Simpan dataset
df.to_csv('student_dataset.csv', index=False)

print(f'✅ Dataset berhasil dibuat!')
print(f'Jumlah data  : {df.shape[0]}')
print(f'Jumlah fitur : {df.shape[1]-1}')
print(f'\nDistribusi label:')
counts = df['pass'].value_counts()
print(f'  Lulus (1)      : {counts[1]} ({counts[1]/n*100:.1f}%)')
print(f'  Tidak Lulus (0): {counts[0]} ({counts[0]/n*100:.1f}%)')
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
print('=== INFO DATASET ===')
print(df.info())
print('\n=== STATISTIK DESKRIPTIF ===')
df.describe()

In [ ]:
# Distribusi label
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
counts = df['pass'].value_counts()
plt.bar(['Tidak Lulus', 'Lulus'], [counts[0], counts[1]],
        color=['#E74C3C', '#2ECC71'], edgecolor='white')
plt.title('Distribusi Label Kelulusan', fontweight='bold')
plt.ylabel('Jumlah Siswa')
for i, v in enumerate([counts[0], counts[1]]):
    plt.text(i, v+3, str(v), ha='center', fontweight='bold')

plt.subplot(1, 3, 2)
plt.hist(df['absences'], bins=20, color='steelblue', edgecolor='white')
plt.title('Distribusi Absensi', fontweight='bold')
plt.xlabel('Jumlah Absensi')

plt.subplot(1, 3, 3)
plt.hist(df['studytime'], bins=4, color='#9B59B6', edgecolor='white')
plt.title('Distribusi Waktu Belajar', fontweight='bold')
plt.xlabel('Waktu Belajar (1-4)')

plt.tight_layout()
plt.savefig('eda_distribusi.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Grafik EDA disimpan')

In [ ]:
# Heatmap korelasi
numeric_cols = df.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(12, 8))
corr = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Heatmap Korelasi Fitur', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('heatmap_korelasi.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Heatmap disimpan')

## 4. Preprocessing

In [ ]:
# Encoding kolom kategorikal
df_enc = df.copy()
cat_cols = df_enc.select_dtypes(include='object').columns
le = LabelEncoder()
for col in cat_cols:
    df_enc[col] = le.fit_transform(df_enc[col])

# Pisah fitur dan target
X = df_enc.drop(columns=['pass'])
y = df_enc['pass']

# Split 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Normalisasi
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print('✅ Preprocessing selesai!')
print(f'Data Training : {X_train.shape[0]} sampel')
print(f'Data Testing  : {X_test.shape[0]} sampel')
print(f'Jumlah Fitur  : {X.shape[1]}')

## 5. Training Model

In [ ]:
models = {
    'Random Forest'       : RandomForestClassifier(n_estimators=100, random_state=42),
    'Decision Tree'       : DecisionTreeClassifier(max_depth=5, random_state=42),
    'Logistic Regression' : LogisticRegression(max_iter=1000, random_state=42)
}

results = {}
print('=== TRAINING MODEL ===')
for name, model in models.items():
    model.fit(X_train_sc, y_train)
    y_pred = model.predict(X_test_sc)
    acc    = accuracy_score(y_test, y_pred)
    cv     = cross_val_score(model, X_train_sc, y_train, cv=5)
    results[name] = {'model': model, 'y_pred': y_pred, 'accuracy': acc,
                     'cv_mean': cv.mean(), 'cv_std': cv.std()}
    print(f'✅ {name:22s}: Acc={acc:.4f} | CV={cv.mean():.4f}±{cv.std():.4f}')

## 6. Evaluasi Model

In [ ]:
# Perbandingan akurasi
names = list(results.keys())
accs  = [results[m]['accuracy'] for m in names]
cvs   = [results[m]['cv_mean']  for m in names]
stds  = [results[m]['cv_std']   for m in names]
x = np.arange(len(names))

plt.figure(figsize=(9, 5))
b1 = plt.bar(x-0.18, accs, 0.35, label='Test Accuracy', color='#3498DB')
b2 = plt.bar(x+0.18, cvs,  0.35, yerr=stds, label='CV Accuracy', color='#E67E22', capsize=5)
for b in list(b1)+list(b2):
    plt.text(b.get_x()+b.get_width()/2, b.get_height()+0.005,
             f'{b.get_height():.3f}', ha='center', fontsize=9, fontweight='bold')
plt.xticks(x, names)
plt.ylim(0.5, 1.0)
plt.ylabel('Akurasi')
plt.title('Perbandingan Akurasi Model', fontweight='bold')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('perbandingan_model.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Grafik perbandingan disimpan')

In [ ]:
# Confusion Matrix
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    ConfusionMatrixDisplay(cm, display_labels=['Tidak Lulus','Lulus']).plot(
        ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{name}\nAcc: {res["accuracy"]:.3f}', fontweight='bold')
plt.suptitle('Confusion Matrix - Semua Model', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Confusion Matrix disimpan')

In [ ]:
# ROC Curve
plt.figure(figsize=(7, 6))
for (name, res), color in zip(results.items(), ['#3498DB','#E67E22','#2ECC71']):
    proba = res['model'].predict_proba(X_test_sc)[:,1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={auc:.3f})')
plt.plot([0,1],[0,1],'k--', lw=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve', fontweight='bold')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ ROC Curve disimpan')

In [ ]:
# Feature Importance
rf = results['Random Forest']['model']
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True).tail(15)
plt.figure(figsize=(8, 6))
plt.barh(imp.index, imp.values, color='#3498DB')
plt.xlabel('Importance Score')
plt.title('Top 15 Fitur Terpenting (Random Forest)', fontweight='bold')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Feature importance disimpan')

In [ ]:
# Classification Report model terbaik
best = max(results, key=lambda x: results[x]['accuracy'])
print(f'=== CLASSIFICATION REPORT: {best} ===')
print(classification_report(y_test, results[best]['y_pred'],
                             target_names=['Tidak Lulus','Lulus']))

## 7. Demo Prediksi

In [ ]:
def prediksi_siswa(data):
    df_in = pd.DataFrame([data])
    for col in X.columns:
        if col not in df_in.columns:
            df_in[col] = 0
    df_in = df_in[X.columns]
    scaled = scaler.transform(df_in)
    pred  = rf.predict(scaled)[0]
    proba = rf.predict_proba(scaled)[0][1]
    print(f'Hasil  : {"✅ LULUS" if pred==1 else "❌ TIDAK LULUS"}')
    print(f'Prob.  : {proba:.1%}')
    return pred, proba

print('=== DEMO PREDIKSI SISWA BARU ===')
contoh = {
    'age':15, 'Medu':3, 'Fedu':2, 'traveltime':1,
    'studytime':3, 'failures':0, 'famrel':4,
    'freetime':3, 'goout':2, 'Dalc':1,
    'Walc':2, 'health':4, 'absences':3,
    'sex':0, 'address':1, 'Pstatus':1,
    'schoolsup':0, 'famsup':1, 'paid':0, 'internet':1
}
prediksi_siswa(contoh)

## 8. Ringkasan Hasil

In [ ]:
print('='*55)
print('           RINGKASAN HASIL PROYEK')
print('='*55)
print(f'Dataset   : Sintetis Student Performance')
print(f'Jumlah Data : {len(df)} siswa')
print(f'Fitur     : {X.shape[1]} fitur')
print(f'Target    : Lulus / Tidak Lulus')
print()
print('--- Performa Model ---')
for name, res in results.items():
    mark = '⭐' if name == best else '  '
    print(f'{mark} {name:22s}: {res["accuracy"]:.4f}')
print()
print(f'✅ Model Terbaik  : {best}')
print(f'✅ Akurasi Terbaik: {results[best]["accuracy"]*100:.2f}%')
print('='*55)
print('\nFile yang dihasilkan:')
for f in ['student_dataset.csv','confusion_matrix.png',
          'roc_curve.png','feature_importance.png',
          'perbandingan_model.png','eda_distribusi.png',
          'heatmap_korelasi.png']:
    print(f'  📄 {f}')